<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/11_train_image_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent storage
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Configure Git remote access
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "your_email@example.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Clone or pull repo
%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

# 4. Inject module search path
src_str = str(SRC_DIR.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)

# 5. Install dependencies
!pip install -q transformers pyarrow redivis scikit-learn

print("=" * 60)
print(f"Working Directory:   {os.getcwd()}")
print(f"Module Search Path:  {src_str}")
print("Environment configuration complete.")
print("=" * 60)

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 148 (delta 61), reused 95 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 518.18 KiB | 8.93 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/DP-MMFL
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.2/862.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.4/407.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 11.5 MB/s eta 0:00:00
Working Directory:   /content/DP-MMFL
Module Search Path:  /content/DP-MMFL/src
Environment configuration complete.


In [ ]:
import os
import shutil
from pathlib import Path
from io import BytesIO
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
import redivis

# 1. Configure paths
MANIFEST_PATH = Path("/content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet")
SOURCE_CACHE = Path("/content/dp_mmfl_image_cache/train")
CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT_256.mkdir(parents=True, exist_ok=True)

# 2. Select the identical deterministic 1,000 frontal samples
manifest = pd.read_parquet(MANIFEST_PATH)
cache_manifest = (
    manifest[(manifest["experiment_split"] == "train") & (manifest["frontal_lateral"] == "Frontal")]
    .sort_values("sample_id")
    .head(1000)
    .reset_index(drop=True)
)

existing_256 = list(CACHE_ROOT_256.glob("*.png"))
print(f"Existing 256x256 images: {len(existing_256)}")

# 3. Populate cache if not already at 1,000
if len(existing_256) < 1000:
    # Check if original raw cache exists locally
    source_exists = SOURCE_CACHE.exists() and len(list(SOURCE_CACHE.glob("*.png"))) == 1000

    if source_exists:
        print("Populating 256x256 cache from local source cache...")
        for row in tqdm(cache_manifest.itertuples(), total=len(cache_manifest)):
            dst_path = CACHE_ROOT_256 / f"{int(row.sample_id):07d}.png"
            if dst_path.exists():
                continue
            src_path = SOURCE_CACHE / f"{int(row.sample_id):07d}.png"
            img = Image.open(src_path).convert("L")
            img = img.resize((256, 256), Image.Resampling.BILINEAR)
            img.save(dst_path, format="PNG")
    else:
        print("Connecting to Redivis to stream and resize 1,000 images directly to 256x256...")
        table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")
        for row in tqdm(cache_manifest.itertuples(), total=len(cache_manifest)):
            dst_path = CACHE_ROOT_256 / f"{int(row.sample_id):07d}.png"
            if dst_path.exists():
                continue

            png_path = str(row.path_to_image)
            if png_path.startswith("train/"):
                png_path = png_path[len("train/"):]
            if png_path.endswith(".jpg"):
                png_path = png_path[:-4] + ".png"

            file = table.file(png_path)
            data = file.read(as_text=False)
            img = Image.open(BytesIO(data)).convert("L")
            img = img.resize((256, 256), Image.Resampling.BILINEAR)
            img.save(dst_path, format="PNG")

total_ready = len(list(CACHE_ROOT_256.glob("*.png")))
print(f"\nCache generation complete: {total_ready}/1000 images ready at {CACHE_ROOT_256}")
assert total_ready == 1000, f"Expected 1,000 images, got {total_ready}"

Existing 256x256 images: 0
Connecting to Redivis to stream and resize 1,000 images directly to 256x256...


  0%|          | 0/1000 [00:00<?, ?it/s]

Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=9324b1eac9c56736a6567216e28a006e

Cache generation complete: 1000/1000 images ready at /content/dp_mmfl_image_cache_256/train


In [ ]:
from pathlib import Path
import torch
from dp_mmfl.utils.seed import set_seed
from dp_mmfl.data.labels import TARGET_COLUMNS

SEED = 42
set_seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"
CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Project Root:      ", PROJECT_ROOT)
print("Manifest Path:     ", MANIFEST_PATH)
print("Manifest Exists:   ", MANIFEST_PATH.exists())
print("256 Cache Exists:  ", CACHE_ROOT_256.exists())
print("Device:            ", device)
if torch.cuda.is_available():
    print("GPU Name:          ", torch.cuda.get_device_name(0))

assert MANIFEST_PATH.exists(), f"Missing manifest at {MANIFEST_PATH}"
assert CACHE_ROOT_256.exists(), f"Missing cache directory at {CACHE_ROOT_256}"

Project Root:       /content/drive/MyDrive/DP-MMFL
Manifest Path:      /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Manifest Exists:    True
256 Cache Exists:   True
Device:             cuda:0
GPU Name:           Tesla T4


In [ ]:
import json

EXPERIMENT_ROOT = PROJECT_ROOT / "results" / "E1_image_baseline"
CHECKPOINT_DIR = EXPERIMENT_ROOT / "checkpoints"
METRICS_DIR = EXPERIMENT_ROOT / "metrics"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment directory: ", EXPERIMENT_ROOT)
print("Checkpoint directory: ", CHECKPOINT_DIR)
print("Metrics directory:    ", METRICS_DIR)

Experiment directory:  /content/drive/MyDrive/DP-MMFL/results/E1_image_baseline
Checkpoint directory:  /content/drive/MyDrive/DP-MMFL/results/E1_image_baseline/checkpoints
Metrics directory:     /content/drive/MyDrive/DP-MMFL/results/E1_image_baseline/metrics


In [ ]:
%%writefile /content/DP-MMFL/src/dp_mmfl/data/preprocessed_cached_dataset.py
from pathlib import Path
from typing import Sequence, Union
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from dp_mmfl.data.labels import TARGET_COLUMNS


class PreprocessedCachedImageDataset(Dataset):
    """
    Loads pre-resized (256x256) cached grayscale images and applies dynamic augmentations/crops.
    Sanitizes multi-label pathology targets (NaN -> 0.0) gated by U-Ignore masks.
    """

    def __init__(
        self,
        manifest_path: Union[str, Path],
        cache_root: Union[str, Path],
        sample_ids: Sequence[int],
        image_transform=None,
    ):
        manifest = pd.read_parquet(manifest_path)

        manifest = manifest[
            manifest["sample_id"].isin(sample_ids)
        ].copy()

        # Preserve strict order of sample_ids
        order = {
            int(sample_id): i
            for i, sample_id in enumerate(sample_ids)
        }

        manifest["_order"] = manifest["sample_id"].map(order)
        manifest = (
            manifest
            .sort_values("_order")
            .drop(columns="_order")
            .reset_index(drop=True)
        )

        self.manifest = manifest
        self.cache_root = Path(cache_root)
        self.image_transform = image_transform

    def __len__(self) -> int:
        return len(self.manifest)

    def __getitem__(self, index: int):
        row = self.manifest.iloc[index]

        image_path = (
            self.cache_root /
            f"{int(row['sample_id']):07d}.png"
        )

        if not image_path.exists():
            raise FileNotFoundError(
                f"Missing cached image: {image_path}"
            )

        image = Image.open(image_path).convert("L")

        if self.image_transform is not None:
            image = self.image_transform(image)

        labels = torch.tensor(
            [
                0.0 if pd.isna(row[f"target_{label}"])
                else float(row[f"target_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        label_mask = torch.tensor(
            [
                float(row[f"mask_{label}"])
                for label in TARGET_COLUMNS
            ],
            dtype=torch.float32,
        )

        return {
            "image": image,
            "labels": labels,
            "label_mask": label_mask,
            "sample_id": row["sample_id"],
            "patient_id": row["deid_patient_id"],
        }

Writing /content/DP-MMFL/src/dp_mmfl/data/preprocessed_cached_dataset.py


In [ ]:
# Commit module to Git
%cd /content/DP-MMFL
!git add src/dp_mmfl/data/preprocessed_cached_dataset.py
!git commit -m "add PreprocessedCachedImageDataset module"
!git push origin main

/content/DP-MMFL
[main 71a10d2] add PreprocessedCachedImageDataset module
 1 file changed, 91 insertions(+)
 create mode 100644 src/dp_mmfl/data/preprocessed_cached_dataset.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (6/6), 1.35 KiB | 1.35 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   6671b47..71a10d2  main -> main


In [ ]:
import json
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. Inspect cached sample IDs present on disk
cached_files = sorted(CACHE_ROOT_256.glob("*.png"))
cached_sample_ids = [int(p.stem) for p in cached_files]
print("Total cached images on disk:", len(cached_sample_ids))
assert len(cached_sample_ids) > 0, "No cached images found!"

# 2. Filter manifest to match cached samples
manifest = pd.read_parquet(MANIFEST_PATH)
cached_manifest = manifest[manifest["sample_id"].isin(cached_sample_ids)].reset_index(drop=True)

# 3. Patient-disjoint split (80% train, 20% val)
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(
        cached_manifest,
        groups=cached_manifest["deid_patient_id"],
    )
)

train_sample_ids = cached_manifest.iloc[train_idx]["sample_id"].astype(int).tolist()
val_sample_ids = cached_manifest.iloc[val_idx]["sample_id"].astype(int).tolist()

train_patients = set(cached_manifest.iloc[train_idx]["deid_patient_id"])
val_patients = set(cached_manifest.iloc[val_idx]["deid_patient_id"])

print("--- Split Profile ---")
print("Train images:       ", len(train_sample_ids))
print("Validation images:  ", len(val_sample_ids))
print("Train patients:     ", len(train_patients))
print("Validation patients:", len(val_patients))
print("Patient Overlap:    ", len(train_patients & val_patients))

assert len(train_patients & val_patients) == 0, "Data leakage detected between train and val!"

# 4. Save split IDs to Google Drive for reproducibility
split_record = {
    "train_sample_ids": train_sample_ids,
    "val_sample_ids": val_sample_ids,
    "train_patients": list(train_patients),
    "val_patients": list(val_patients),
}
with open(METRICS_DIR / "split_info.json", "w") as f:
    json.dump(split_record, f, indent=2)
print("Saved split configuration to:", METRICS_DIR / "split_info.json")

Total cached images on disk: 1000
--- Split Profile ---
Train images:        835
Validation images:   165
Train patients:      269
Validation patients: 68
Patient Overlap:     0
Saved split configuration to: /content/drive/MyDrive/DP-MMFL/results/E1_image_baseline/metrics/split_info.json


In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset

train_transform = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=CACHE_ROOT_256,
    sample_ids=train_sample_ids,
    image_transform=train_transform,
)

val_dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=CACHE_ROOT_256,
    sample_ids=val_sample_ids,
    image_transform=eval_transform,
)

BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

print(f"Train Dataset: {len(train_dataset)} samples | {len(train_loader)} batches")
print(f"Val Dataset:   {len(val_dataset)} samples | {len(val_loader)} batches")

Train Dataset: 835 samples | 27 batches
Val Dataset:   165 samples | 6 batches


In [ ]:
from dp_mmfl.models.image_model import ResNet18Classifier

model = ResNet18Classifier(
    num_classes=13,
    pretrained=True,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

initial_checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 0,
    "experiment": "E1_image_baseline_development",
    "seed": SEED,
    "target_columns": TARGET_COLUMNS,
    "train_sample_ids": train_sample_ids,
    "val_sample_ids": val_sample_ids,
}

torch.save(initial_checkpoint, CHECKPOINT_DIR / "initial.pt")
print("Initial checkpoint saved:", (CHECKPOINT_DIR / "initial.pt").exists())

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 167MB/s]


Initial checkpoint saved: True


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
from dp_mmfl.evaluation.losses import masked_bce_with_logits


def evaluate(model, loader, device):
    """
    Runs evaluation over the validation loader.
    Computes average masked BCE loss and per-class / macro AUROC for labels with valid binary distribution.
    """
    model.eval()
    total_loss = 0.0
    total_valid_labels = 0

    all_logits = []
    all_targets = []
    all_masks = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)
            mask = batch["label_mask"].to(device, non_blocking=True)

            logits = model(images)
            loss = masked_bce_with_logits(logits, labels, mask)

            batch_valid = mask.sum().item()
            total_loss += loss.item() * batch_valid
            total_valid_labels += batch_valid

            all_logits.append(logits.cpu())
            all_targets.append(labels.cpu())
            all_masks.append(mask.cpu())

    avg_loss = total_loss / max(total_valid_labels, 1)

    all_probs = torch.sigmoid(torch.cat(all_logits, dim=0)).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_masks = torch.cat(all_masks, dim=0).numpy()

    # Calculate per-class AUROC (ignoring masked entries)
    per_class_auroc = {}
    valid_aurocs = []

    for i, col in enumerate(TARGET_COLUMNS):
        col_mask = all_masks[:, i] == 1.0
        y_true = all_targets[col_mask, i]
        y_pred = all_probs[col_mask, i]

        # Only compute AUROC if both positive and negative cases exist in validation split
        if len(np.unique(y_true)) == 2:
            score = roc_auc_score(y_true, y_pred)
            per_class_auroc[col] = float(score)
            valid_aurocs.append(score)
        else:
            per_class_auroc[col] = None

    macro_auroc = float(np.mean(valid_aurocs)) if valid_aurocs else None

    return avg_loss, macro_auroc, per_class_auroc

In [ ]:
import time

NUM_EPOCHS = 3
history = []
best_val_loss = float("inf")

print(f"Starting E1 Baseline Training: {NUM_EPOCHS} epochs on {device.type.upper()}\n")

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.perf_counter()

    # --- Training Phase ---
    model.train()
    train_loss_accum = 0.0
    train_valid_labels = 0

    for batch in train_loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        mask = batch["label_mask"].to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = masked_bce_with_logits(logits, labels, mask)
        loss.backward()
        optimizer.step()

        batch_valid = mask.sum().item()
        train_loss_accum += loss.item() * batch_valid
        train_valid_labels += batch_valid

    avg_train_loss = train_loss_accum / max(train_valid_labels, 1)

    # --- Validation Phase ---
    val_loss, val_macro_auroc, per_class_auroc = evaluate(model, val_loader, device)
    epoch_duration = time.perf_counter() - epoch_start

    auroc_str = f"{val_macro_auroc:.4f}" if val_macro_auroc is not None else "N/A"
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} [{epoch_duration:.1f}s] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Macro AUROC: {auroc_str}"
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "val_macro_auroc": val_macro_auroc,
        "per_class_auroc": per_class_auroc,
        "duration_sec": epoch_duration,
    }
    history.append(epoch_record)

    # Checkpoint per epoch
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": avg_train_loss,
            "val_loss": val_loss,
        },
        CHECKPOINT_DIR / f"epoch_{epoch:02d}.pt",
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model.pt")
        print(f"  --> Saved new best model (Val Loss: {val_loss:.4f})")

# Save training history JSON
with open(METRICS_DIR / "e1_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nE1 training run complete. All checkpoints and metrics saved to Google Drive.")

Starting E1 Baseline Training: 3 epochs on CUDA

Epoch 01/03 [8.5s] | Train Loss: 0.5574 | Val Loss: 0.5163 | Val Macro AUROC: 0.4695
  --> Saved new best model (Val Loss: 0.5163)
Epoch 02/03 [3.5s] | Train Loss: 0.3861 | Val Loss: 0.5012 | Val Macro AUROC: 0.4890
  --> Saved new best model (Val Loss: 0.5012)
Epoch 03/03 [3.6s] | Train Loss: 0.3030 | Val Loss: 0.4956 | Val Macro AUROC: 0.5392
  --> Saved new best model (Val Loss: 0.4956)

E1 training run complete. All checkpoints and metrics saved to Google Drive.
